In [1]:
import os
from dotenv import load_dotenv
from langgraph.store.postgres import PostgresStore
from typing import Final,Tuple

load_dotenv(override=True)

DB_URL = os.getenv("DB_URL")

with PostgresStore.from_conn_string(DB_URL) as store:
    #1. 创建长期记忆存储的表格
    store.setup()
    #2. 构建永久记忆数据
    #2.1 构建命名空间
    USERS_NS:Final[Tuple[str]] = ("users",)
    PREFERENCES_KEY:Final[str] = "preferences"

    namespace1 = (*USERS_NS,"Alice")
    namespace2 = (*USERS_NS,"Bob")
    namespace3 = (*USERS_NS,"Black")

    value1 = {
        "course":"计算机组成原理",
        "sports":"跑步",
        "food":"紫光园奶皮子酸奶"
    }

    value2 = {
        "course": "数字电路与模拟电路",
        "sports": "跑步",
        "food": "奶皮子糖葫芦"
    }

    value3 = {
        "course": "数字电路与模拟电路",
        "sports": "羽毛球",
        "food": "紫光园奶皮子酸奶"
    }

    #3. 写入永久记忆数据
    store.put(namespace1,PREFERENCES_KEY,value1)
    store.put(namespace2,PREFERENCES_KEY,value2)
    store.put(namespace3,PREFERENCES_KEY,value3)

    for item in store.search(USERS_NS):
        print(item)


Item(namespace=['users', 'Black'], key='preferences', value={'food': '紫光园奶皮子酸奶', 'course': '数字电路与模拟电路', 'sports': '羽毛球'}, created_at='2026-07-09T12:02:08.994560+08:00', updated_at='2026-07-09T14:50:06.575525+08:00', score=None)
Item(namespace=['users', 'Bob'], key='preferences', value={'food': '奶皮子糖葫芦', 'course': '数字电路与模拟电路', 'sports': '跑步'}, created_at='2026-07-09T12:02:08.994224+08:00', updated_at='2026-07-09T14:50:06.575090+08:00', score=None)
Item(namespace=['users', 'Alice'], key='preferences', value={'food': '紫光园奶皮子酸奶', 'course': '计算机组成原理', 'sports': '跑步'}, created_at='2026-07-09T12:02:08.989782+08:00', updated_at='2026-07-09T14:50:06.573588+08:00', score=None)


In [1]:
from typing import TypedDict, Annotated, Literal

from dotenv import  load_dotenv
from langchain_core.messages import SystemMessage,HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph,START,END
from langgraph.runtime import Runtime
from loguru import logger
from langgraph.checkpoint.postgres import  PostgresSaver
from langgraph.graph.message import MessagesState

load_dotenv(override=True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 声明全局状态
class OverAllState(MessagesState):
    username:str
    user_input:str
    output:str
    preferences:dict[str,str]

#2. 声明节点
#2.1 路由函数 如果状态中没有用户偏好 则从长期记忆中查询,否则直接执行大模型节点
def router(state:OverAllState) -> Literal["check_preference_node","llm_node"]:
    if not state.get("preferences"):
        logger.info("需要从长期记忆中读取用户偏好")
        return "check_preference_node"
    logger.info("用户偏好已经存在,不需要查询")
    return "llm_node"

#2.2 检查长期记忆节点
def check_preference_node(state:OverAllState,runtime:Runtime) -> OverAllState:
    #1. 拼接命名空间
    username = state["username"]
    namespace = (*USERS_NS,username)
    key = PREFERENCES_KEY
    #2. 获取长期记忆数据
    run_store = runtime.store
    run_item = run_store.get(namespace,key)

    if not item:
        logger.warning("长期记忆中没有{}的偏好数据",username)
        return {}

    logger.info("长期记忆中保存的{}的偏好数据是{}",username,run_item.value)
    return {
        "preferences":item.value
    }

def llm_node(state:OverAllState) -> OverAllState:
    # 检查是否存在长期记忆
    preference = state.get("preferences",{})
    user_input = state["user_input"]
    human_prompt = (f"这是用户的偏好: {preference}\n,这是用户的需求:{user_input}")
    system_prompt = "请根据用户的偏好解决用户的需求"

    messages: list[SystemMessage|HumanMessage|AIMessage|ToolMessage] = [SystemMessage(content=system_prompt)] if not state.get("messages",[]) else state["messages"]

    model_response = model.invoke(messages + [HumanMessage(content=human_prompt)])

    output = model_response.content

    return {
        "messages":messages+[HumanMessage(content=human_prompt),model_response],
        "output":output
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("check_preference_node",check_preference_node)
builder.add_node("llm_node",llm_node)

builder.add_conditional_edges(START,router,path_map=["check_preference_node","llm_node"])
builder.add_edge("check_preference_node","llm_node")
builder.add_edge("llm_node",END)

#4. 构建长期记忆和短期记忆
with PostgresStore.from_conn_string(DB_URL) as store,\
    PostgresSaver.from_conn_string(DB_URL) as checkpointer :
    # 幂等操作  多次执行不会重新创建表格 不会删除数据库中的数据
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer,store=store)

    from IPython.display import display
    display(graph)

    config = {
        "configurable":{"thread_id":"777"}
    }

    res = graph.invoke({"username":"Alice","user_input":"我有点无聊,和我聊聊天吧"},config=config)

    print('=' * 50)
    print(res)

    res1 = graph.invoke({"username":"Alice","user_input":"推荐一下酸奶"},config=config)

    print('=' * 50)
    print(res1)



NameError: name 'PostgresStore' is not defined